In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
!pip install gpboost
import gpboost as gpb
from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, roc_auc_score
)
from scipy.stats import chi2_contingency
from scipy.stats import kruskal, wilcoxon



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


RADAR Analysis

Gradient Boosting Regressor 

- for prediction of continuous phq8 variable
- phq8 then transformed into categorical data to calculate metrics of the regressor

In [8]:

# -----------------------------
# Load and prepare data
# -----------------------------
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/radar_model_dataset_raw_features.csv")
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(dataset)

feature_cols = df.iloc[:, 12:-1].columns.tolist()
df = df.dropna(subset=feature_cols + ["phq8_score", "participant_id"]).copy()

# Binary label for classification-style metrics
df["depressed"] = (df["phq8_score"] >= 10).astype(int)

X = df[feature_cols].values
y = df["phq8_score"].values
y_bin = df["depressed"].values
groups = df["participant_id"].values

# -----------------------------
# Train / test split by participant
# -----------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_full, X_test = X[train_idx], X[test_idx]
y_train_full, y_test = y[train_idx], y[test_idx]
y_bin_train_full, y_bin_test = y_bin[train_idx], y_bin[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full)
X_test = scaler.transform(X_test)

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train participants:", len(np.unique(groups_train)))
print("Test participants:", len(np.unique(groups_test)))

# -----------------------------
# Statistical test: train vs test PHQ-8 balance
# -----------------------------
train_test_stat, train_test_p = kruskal(y_train_full, y_test)

train_test_balance_df = pd.DataFrame([{
    "comparison": "train_vs_test",
    "train_n": len(y_train_full),
    "test_n": len(y_test),
    "train_mean_phq8": np.mean(y_train_full),
    "train_median_phq8": np.median(y_train_full),
    "test_mean_phq8": np.mean(y_test),
    "test_median_phq8": np.median(y_test),
    "kruskal_H": train_test_stat,
    "p_value": train_test_p
}])

print("\nTrain/Test PHQ-8 balance:")
print(train_test_balance_df)

# -----------------------------
# Cross-validation on training set only
# -----------------------------
gkf = GroupKFold(n_splits=5)

cv_results = []
cv_fold_phq8 = []

for fold, (cv_train_idx, cv_val_idx) in enumerate(
    gkf.split(X_train_full, y_train_full, groups=groups_train), start=1
):
    X_train, X_val = X_train_full[cv_train_idx], X_train_full[cv_val_idx]
    y_train, y_val = y_train_full[cv_train_idx], y_train_full[cv_val_idx]
    y_bin_val = y_bin_train_full[cv_val_idx]

    cv_fold_phq8.append(y_val)

    model = GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    preds_bin = (preds >= 10).astype(int)

    # Paired Wilcoxon: |y - pred_GBR| vs |y - train-mean| (per-sample, same val rows)
    mean_train = float(np.mean(y_train))
    pred_mean = np.full_like(y_val, mean_train, dtype=float)
    e_gbr = np.abs(y_val - preds)
    e_base = np.abs(y_val - pred_mean)
    if len(e_gbr) >= 2 and not np.allclose(e_gbr, e_base):
        w_cv = float(wilcoxon(e_gbr, e_base, zero_method="wilcox", mode="auto").pvalue)
    else:
        w_cv = float("nan")
    print(f"  Wilcoxon p (|y-ŷ| GBR vs |y-ȳ_train|) fold {fold}:", w_cv)

    cv_results.append({
        "fold": fold,
        "mae": mean_absolute_error(y_val, preds),
        "rmse": np.sqrt(mean_squared_error(y_val, preds)),
        "r2": r2_score(y_val, preds),
        "accuracy": accuracy_score(y_bin_val, preds_bin),
        "f1": f1_score(y_bin_val, preds_bin),
        "roc_auc": roc_auc_score(y_bin_val, preds),
        "wilcoxon_p_vs_train_mean_baseline": w_cv
    })

cv_results_df = pd.DataFrame(cv_results)

print("\nCV fold results:")
print(cv_results_df)

# -----------------------------
# Statistical test: PHQ-8 balance across CV folds
# -----------------------------
cv_stat, cv_p = kruskal(*cv_fold_phq8)

cv_balance_rows = []
for i, vals in enumerate(cv_fold_phq8, start=1):
    cv_balance_rows.append({
        "fold": i,
        "n_rows": len(vals),
        "mean_phq8": np.mean(vals),
        "median_phq8": np.median(vals)
    })

cv_balance_df = pd.DataFrame(cv_balance_rows)
cv_balance_summary_df = pd.DataFrame([{
    "comparison": "cv_folds",
    "kruskal_H": cv_stat,
    "p_value": cv_p
}])

print("\nCV fold PHQ-8 balance:")
print(cv_balance_df)
print("\nCV fold Kruskal-Wallis test:")
print(cv_balance_summary_df)

cv_summary_df = pd.DataFrame([{
    "subset": "cv_train",
    "n_rows": len(train_idx),
    "n_depressed": int(y_bin_train_full.sum()),
    "n_control": int((y_bin_train_full == 0).sum()),
    "mae_mean": cv_results_df["mae"].mean(),
    "mae_std": cv_results_df["mae"].std(),
    "rmse_mean": cv_results_df["rmse"].mean(),
    "rmse_std": cv_results_df["rmse"].std(),
    "r2_mean": cv_results_df["r2"].mean(),
    "r2_std": cv_results_df["r2"].std(),
    "accuracy_mean": cv_results_df["accuracy"].mean(),
    "accuracy_std": cv_results_df["accuracy"].std(),
    "f1_mean": cv_results_df["f1"].mean(),
    "f1_std": cv_results_df["f1"].std(),
    "roc_auc_mean": cv_results_df["roc_auc"].mean(),
    "roc_auc_std": cv_results_df["roc_auc"].std(),
}])

print("\nCV summary:")
print(cv_summary_df)

# -----------------------------
# Final model: train on all training data, evaluate on held-out test set
# -----------------------------
final_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

final_model.fit(X_train_full, y_train_full)
test_preds = final_model.predict(X_test)
test_preds_bin = (test_preds >= 10).astype(int)

test_summary_df = pd.DataFrame([{
    "subset": "held_out_test",
    "n_rows": len(test_idx),
    "n_depressed": int(y_bin_test.sum()),
    "n_control": int((y_bin_test == 0).sum()),
    "mae": mean_absolute_error(y_test, test_preds),
    "rmse": np.sqrt(mean_squared_error(y_test, test_preds)),
    "r2": r2_score(y_test, test_preds),
    "accuracy": accuracy_score(y_bin_test, test_preds_bin),
    "f1": f1_score(y_bin_test, test_preds_bin),
    "roc_auc": roc_auc_score(y_bin_test, test_preds)
}])

print("\nHeld-out test results:")
print(test_summary_df)

baseline_test = np.full_like(y_test, float(np.mean(y_train_full)), dtype=float)
e_gbr_t = np.abs(y_test - test_preds)
e_b_t = np.abs(y_test - baseline_test)
if len(e_gbr_t) >= 2 and not np.allclose(e_gbr_t, e_b_t):
    w_test = float(wilcoxon(e_gbr_t, e_b_t, zero_method="wilcox", mode="auto").pvalue)
else:
    w_test = float("nan")
print("\nWilcoxon p (held-out, paired |y-ŷ_GBR| vs |y-ȳ_train|):", w_test)
if w_test < 0.05:
    print("  Significant difference from train-mean baseline (alpha=0.05) on this test set.")
else:
    print("  No significant difference from train-mean baseline at alpha=0.05 (improvement may be chance on this split).")

pd.DataFrame([{"wilcoxon_p_vs_train_mean_baseline_heldout": w_test}]).to_csv(
    RESULTS_PATH / "Radar_xgb_wilcoxon_heldout.csv", index=False
)

# -----------------------------
# Save results
# -----------------------------
cv_results_df.to_csv(RESULTS_PATH / "Radar_xgb_cv_folds.csv", index=False)
cv_summary_df.to_csv(RESULTS_PATH / "Radar_xgb_cv_summary.csv", index=False)
test_summary_df.to_csv(RESULTS_PATH / "Radar_xgb_test_summary.csv", index=False)

train_test_balance_df.to_csv(RESULTS_PATH / "Radar_xgb_train_test_balance.csv", index=False)
cv_balance_df.to_csv(RESULTS_PATH / "Radar_xgb_cv_fold_balance.csv", index=False)
cv_balance_summary_df.to_csv(RESULTS_PATH / "Radar_xgb_cv_fold_balance_test.csv", index=False)

Train rows: 6570
Test rows: 1945
Train participants: 219
Test participants: 55

Train/Test PHQ-8 balance:
      comparison  train_n  test_n  train_mean_phq8  train_median_phq8  \
0  train_vs_test     6570    1945          9.28828                9.0   

   test_mean_phq8  test_median_phq8  kruskal_H       p_value  
0         8.19126               7.0  54.532593  1.528910e-13  
  Wilcoxon p (|y-ŷ| GBR vs |y-ȳ_train|) fold 1: 4.0097956537728345e-07
  Wilcoxon p (|y-ŷ| GBR vs |y-ȳ_train|) fold 2: 0.11716624963266961
  Wilcoxon p (|y-ŷ| GBR vs |y-ȳ_train|) fold 3: 0.035632801831558814
  Wilcoxon p (|y-ŷ| GBR vs |y-ȳ_train|) fold 4: 0.23039087819754778
  Wilcoxon p (|y-ŷ| GBR vs |y-ȳ_train|) fold 5: 0.00032411384080576486

CV fold results:
   fold       mae      rmse        r2  accuracy        f1   roc_auc  \
0     1  5.503637  6.693088  0.022327  0.571537  0.468366  0.620677   
1     2  5.230133  6.099182 -0.102878  0.575342  0.401288  0.532761   
2     3  5.410644  6.474827 -0.062007  0.52

Gradient Boosting Classifier

- phq8 transformed into categorical data (1,0) for binary classification
- multiclass classification?

- Gridsearch CV used to assess the best params for tuning

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

from scipy.stats import chi2_contingency, wilcoxon

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# -----------------------------
# Load and prepare data
# -----------------------------
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/radar_model_dataset_raw_features.csv")
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(dataset)

feature_cols = df.iloc[:, 12:-1].columns.tolist()
df = df.dropna(subset=feature_cols + ["phq8_score", "participant_id"]).copy()

# Binary target
df["depressed"] = (df["phq8_score"] >= 10).astype(int)

X = df[feature_cols].values
y_bin = df["depressed"].values
groups = df["participant_id"].values

# -----------------------------
# Held-out test split by participant
# -----------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y_bin, groups=groups))

X_train_full, X_test = X[train_idx], X[test_idx]
y_train_full, y_test = y_bin[train_idx], y_bin[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full)
X_test = scaler.transform(X_test)

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train participants:", len(np.unique(groups_train)))
print("Test participants:", len(np.unique(groups_test)))

# -----------------------------
# Statistical test: train vs test label balance
# -----------------------------
train_test_table = pd.DataFrame({
    "train": pd.Series(y_train_full).value_counts().sort_index(),
    "test": pd.Series(y_test).value_counts().sort_index()
}).fillna(0)

train_test_table.index = ["control", "depressed"]

chi2_train_test, p_train_test, dof_train_test, expected_train_test = chi2_contingency(train_test_table.T)

train_test_balance_df = pd.DataFrame([{
    "comparison": "train_vs_test",
    "train_n": len(y_train_full),
    "test_n": len(y_test),
    "train_control": int((y_train_full == 0).sum()),
    "train_depressed": int((y_train_full == 1).sum()),
    "test_control": int((y_test == 0).sum()),
    "test_depressed": int((y_test == 1).sum()),
    "chi2": chi2_train_test,
    "p_value": p_train_test
}])

print("\nTrain/Test label distribution:")
print(train_test_table)
print("\nTrain/Test Chi-square test:")
print(train_test_balance_df)

# -----------------------------
# Cross-validation on training set only
# -----------------------------
gkf = GroupKFold(n_splits=5)
cv_results = []
cv_fold_rows = []

for fold, (cv_train_idx, cv_val_idx) in enumerate(
    gkf.split(X_train_full, y_train_full, groups=groups_train), start=1
):
    X_train, X_val = X_train_full[cv_train_idx], X_train_full[cv_val_idx]
    y_train, y_val = y_train_full[cv_train_idx], y_train_full[cv_val_idx]

    counts = pd.Series(y_val).value_counts().sort_index()
    cv_fold_rows.append({
        "fold": fold,
        "control": int(counts.get(0, 0)),
        "depressed": int(counts.get(1, 0)),
        "n_rows": len(y_val)
    })

    model = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    proba = model.predict_proba(X_val)[:, 1]

    # Paired Wilcoxon: |y - p_GBC| vs |y - p_stratified_dummy|
    dummy = DummyClassifier(strategy="stratified", random_state=42)
    dummy.fit(X_train, y_train)
    p_dum = dummy.predict_proba(X_val)[:, 1]
    e1 = np.abs(y_val - proba)
    e2 = np.abs(y_val - p_dum)
    if len(e1) >= 2 and not np.allclose(e1, e2):
        w_p = float(wilcoxon(e1, e2, zero_method="wilcox", mode="auto").pvalue)
    else:
        w_p = float("nan")
    print(f"  Wilcoxon p (|y - p_GBC| vs |y - p_dummy|) fold {fold}:", w_p)

    cv_results.append({
        "fold": fold,
        "accuracy": accuracy_score(y_val, preds),
        "f1": f1_score(y_val, preds),
        "roc_auc": roc_auc_score(y_val, proba),
        "wilcoxon_p_vs_stratified_dummy": w_p
    })

cv_results_df = pd.DataFrame(cv_results)

print("\nCV fold results:")
print(cv_results_df)

# -----------------------------
# Statistical test: label balance across CV folds
# -----------------------------
cv_balance_df = pd.DataFrame(cv_fold_rows).set_index("fold")
chi2_cv, p_cv, dof_cv, expected_cv = chi2_contingency(cv_balance_df[["control", "depressed"]])

cv_balance_summary_df = pd.DataFrame([{
    "comparison": "cv_folds",
    "chi2": chi2_cv,
    "p_value": p_cv
}])

print("\nCV fold label distribution:")
print(cv_balance_df)
print("\nCV fold Chi-square test:")
print(cv_balance_summary_df)

cv_summary_df = pd.DataFrame([{
    "subset": "cv_train",
    "n_rows": len(train_idx),
    "n_depressed": int(y_train_full.sum()),
    "n_control": int((y_train_full == 0).sum()),
    "accuracy_mean": cv_results_df["accuracy"].mean(),
    "accuracy_std": cv_results_df["accuracy"].std(),
    "f1_mean": cv_results_df["f1"].mean(),
    "f1_std": cv_results_df["f1"].std(),
    "roc_auc_mean": cv_results_df["roc_auc"].mean(),
    "roc_auc_std": cv_results_df["roc_auc"].std(),
}])

print("\nCV summary:")
print(cv_summary_df)

# -----------------------------
# Final model on full training set
# -----------------------------
final_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

final_model.fit(X_train_full, y_train_full)

test_preds = final_model.predict(X_test)
test_proba = final_model.predict_proba(X_test)[:, 1]

test_summary_df = pd.DataFrame([{
    "subset": "held_out_test",
    "n_rows": len(test_idx),
    "n_depressed": int(y_test.sum()),
    "n_control": int((y_test == 0).sum()),
    "accuracy": accuracy_score(y_test, test_preds),
    "f1": f1_score(y_test, test_preds),
    "roc_auc": roc_auc_score(y_test, test_proba)
}])

print("\nHeld-out test results:")
print(test_summary_df)

dummy_t = DummyClassifier(strategy="stratified", random_state=42)
dummy_t.fit(X_train_full, y_train_full)
p_dum_t = dummy_t.predict_proba(X_test)[:, 1]
e_gb = np.abs(y_test - test_proba)
e_du = np.abs(y_test - p_dum_t)
if len(e_gb) >= 2 and not np.allclose(e_gb, e_du):
    w_held = float(wilcoxon(e_gb, e_du, zero_method="wilcox", mode="auto").pvalue)
else:
    w_held = float("nan")
print("\nWilcoxon p (held-out, |y - p_GBC| vs |y - p_dummy|):", w_held)
if w_held < 0.05:
    print("  Distributions of abs calibration errors differ from dummy (alpha=0.05).")
else:
    print("  No significant difference from stratified-dummy at alpha=0.05 on this test set (could be chance / split).")

pd.DataFrame([{"wilcoxon_p_vs_stratified_dummy_heldout": w_held}]).to_csv(
    RESULTS_PATH / "radar_xgbc_wilcoxon_heldout.csv", index=False
)

# -----------------------------
# Save results
# -----------------------------
cv_results_df.to_csv(RESULTS_PATH / "radar_xgbc_cv_folds.csv", index=False)
cv_summary_df.to_csv(RESULTS_PATH / "radar_xgbc_cv_summary.csv", index=False)
test_summary_df.to_csv(RESULTS_PATH / "radar_xgbc_test_summary.csv", index=False)

train_test_balance_df.to_csv(RESULTS_PATH / "radar_xgbc_train_test_balance.csv", index=False)
cv_balance_df.to_csv(RESULTS_PATH / "radar_xgbc_cv_fold_balance.csv")
cv_balance_summary_df.to_csv(RESULTS_PATH / "radar_xgbc_cv_fold_balance_test.csv", index=False)

Train rows: 6570
Test rows: 1945
Train participants: 219
Test participants: 55

Train/Test label distribution:
           train  test
control     3625  1278
depressed   2945   667

Train/Test Chi-square test:
      comparison  train_n  test_n  train_control  train_depressed  \
0  train_vs_test     6570    1945           3625             2945   

   test_control  test_depressed       chi2       p_value  
0          1278             667  67.720721  1.883734e-16  
  Wilcoxon p (|y - p_GBC| vs |y - p_dummy|) fold 1: 1.9752215417204264e-06
  Wilcoxon p (|y - p_GBC| vs |y - p_dummy|) fold 2: 0.10266867141776985
  Wilcoxon p (|y - p_GBC| vs |y - p_dummy|) fold 3: 0.793716755698076
  Wilcoxon p (|y - p_GBC| vs |y - p_dummy|) fold 4: 0.0041316882865740125
  Wilcoxon p (|y - p_GBC| vs |y - p_dummy|) fold 5: 0.004892988877607462

CV fold results:
   fold  accuracy        f1   roc_auc  wilcoxon_p_vs_stratified_dummy
0     1  0.576104  0.494096  0.610421                        0.000002
1     2  0.5

GPBoost accounts for group random effects by selecting a source of data clustering.

In this case, ID of the participants served as the source of clustering.

In [10]:

# -----------------------------
# Paths
# -----------------------------
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/radar_model_dataset_raw_features.csv")
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.ensemble import GradientBoostingClassifier
from scipy.stats import wilcoxon

# -----------------------------
# Load and prepare data
# -----------------------------
df = pd.read_csv(dataset)

feature_cols = df.iloc[:, 12:-1].columns.tolist()

df = df.dropna(subset=feature_cols + ["phq8_score", "participant_id"]).copy()
df["depressed"] = (df["phq8_score"] >= 10).astype(int)
df["group_code"] = pd.factorize(df["participant_id"])[0].astype(np.int32)

X = df[feature_cols].values
y = df["depressed"].astype(int).values
groups = df["group_code"].values

# -----------------------------
# Held-out test split by participant
# -----------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_full, X_test = X[train_idx], X[test_idx]
y_train_full, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full)
X_test = scaler.transform(X_test)

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train participants:", len(np.unique(groups_train)))
print("Test participants:", len(np.unique(groups_test)))

# -----------------------------
# Cross-validation on training set only
# -----------------------------
gkf = GroupKFold(n_splits=5)
fold_results = []

for fold, (cv_train_idx, cv_val_idx) in enumerate(
    gkf.split(X_train_full, y_train_full, groups=groups_train), start=1
):
    X_train, X_val = X_train_full[cv_train_idx], X_train_full[cv_val_idx]
    y_train, y_val = y_train_full[cv_train_idx], y_train_full[cv_val_idx]
    group_train = groups_train[cv_train_idx].astype(np.int32)
    group_val = groups_train[cv_val_idx].astype(np.int32)

    gp_model = gpb.GPModel(group_data=group_train)
    train_data = gpb.Dataset(X_train, label=y_train)

    params = {
        "objective": "binary",
        "learning_rate": 0.05,
        "max_depth": 4,
        "verbose": 0
    }

    gpbst = gpb.train(
        params=params,
        train_set=train_data,
        gp_model=gp_model,
        num_boost_round=200
    )

    pred = gpbst.predict(
        data=X_val,
        group_data_pred=group_val,
        predict_var=False
    )

    p_gpboost = pred["response_mean"]
    y_pred = (p_gpboost >= 0.5).astype(int)

    gbc = GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
    )
    gbc.fit(X_train, y_train)
    p_gbc = gbc.predict_proba(X_val)[:, 1]
    e_gbc = np.abs(y_val - p_gbc)
    e_gp = np.abs(y_val - p_gpboost)
    if len(e_gbc) >= 2 and not np.allclose(e_gbc, e_gp):
        w_pair = float(wilcoxon(e_gbc, e_gp, zero_method="wilcox", mode="auto").pvalue)
    else:
        w_pair = float("nan")
    print(f"  Wilcoxon p (|y - p_GBC| vs |y - p_GP|) fold {fold}:", w_pair)

    fold_results.append({
        "fold": fold,
        "accuracy": accuracy_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred),
        "roc_auc": roc_auc_score(y_val, p_gpboost),
        "wilcoxon_p_gbc_vs_gpboost": w_pair
    })

# Fold-level CV results
fold_results_df = pd.DataFrame(fold_results)
print("CV fold-level results:")
print(fold_results_df)

# CV summary
cv_summary_df = pd.DataFrame([{
    "subset": "cv_train",
    "n_rows": len(train_idx),
    "n_depressed": int(y_train_full.sum()),
    "n_control": int((y_train_full == 0).sum()),
    "accuracy_mean": fold_results_df["accuracy"].mean(),
    "accuracy_std": fold_results_df["accuracy"].std(),
    "f1_mean": fold_results_df["f1"].mean(),
    "f1_std": fold_results_df["f1"].std(),
    "roc_auc_mean": fold_results_df["roc_auc"].mean(),
    "roc_auc_std": fold_results_df["roc_auc"].std()
}])

print("\nCV summary:")
print(cv_summary_df)

# -----------------------------
# Final GPBoost model on full training set
# -----------------------------
gp_model_final = gpb.GPModel(group_data=groups_train.astype(np.int32))
train_data_final = gpb.Dataset(X_train_full, label=y_train_full)

params = {
    "objective": "binary",
    "learning_rate": 0.05,
    "max_depth": 4,
    "verbose": 0
}

gpbst_final = gpb.train(
    params=params,
    train_set=train_data_final,
    gp_model=gp_model_final,
    num_boost_round=200
)

test_pred = gpbst_final.predict(
    data=X_test,
    group_data_pred=groups_test.astype(np.int32),
    predict_var=False
)

test_proba = test_pred["response_mean"]
test_preds = (test_proba >= 0.5).astype(int)

test_summary_df = pd.DataFrame([{
    "subset": "held_out_test",
    "n_rows": len(test_idx),
    "n_depressed": int(y_test.sum()),
    "n_control": int((y_test == 0).sum()),
    "accuracy": accuracy_score(y_test, test_preds),
    "f1": f1_score(y_test, test_preds),
    "roc_auc": roc_auc_score(y_test, test_proba)
}])

print("\nHeld-out test results:")
print(test_summary_df)

gbc_test = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
)
gbc_test.fit(X_train_full, y_train_full)
p_gbc_test = gbc_test.predict_proba(X_test)[:, 1]
e1h = np.abs(y_test - p_gbc_test)
e2h = np.abs(y_test - test_proba)
if len(e1h) >= 2 and not np.allclose(e1h, e2h):
    w_held = float(wilcoxon(e1h, e2h, zero_method="wilcox", mode="auto").pvalue)
else:
    w_held = float("nan")
print("\nWilcoxon p (held-out, |y - p_GBC| vs |y - p_GP|):", w_held)
if w_held < 0.05:
    print("  GBC and GPBoost calibration errors differ significantly (alpha=0.05) on this test set.")
else:
    print("  No significant difference between GBC and GPBoost at alpha=0.05 (difference may be chance / test-set specific).")

pd.DataFrame([{"wilcoxon_p_gbc_vs_gpboost_heldout": w_held}]).to_csv(
    RESULTS_PATH / "radar_gpboost_wilcoxon_heldout.csv", index=False
)

# -----------------------------
# Save results
# -----------------------------
fold_results_df.to_csv(RESULTS_PATH / "radar_gpboost_cv_folds.csv", index=False)
cv_summary_df.to_csv(RESULTS_PATH / "radar_gpboost_cv_summary.csv", index=False)
test_summary_df.to_csv(RESULTS_PATH / "radar_gpboost_test_summary.csv", index=False)

Train rows: 6570
Test rows: 1945
Train participants: 219
Test participants: 55
[GPBoost] [Warning] The 'objective' (='binary') for boosting and the 'likelihood' (='gaussian') for the GPModel do not match. It is assumed that the 'objective' for boosting is correctly specified, and the likelihood of the GPModel is changed accordingly. This can be problematic if the GPModel has been pre-trained 
  Wilcoxon p (|y - p_GBC| vs |y - p_GP|) fold 1: 1.2891651054330483e-11
[GPBoost] [Warning] The 'objective' (='binary') for boosting and the 'likelihood' (='gaussian') for the GPModel do not match. It is assumed that the 'objective' for boosting is correctly specified, and the likelihood of the GPModel is changed accordingly. This can be problematic if the GPModel has been pre-trained 
  Wilcoxon p (|y - p_GBC| vs |y - p_GP|) fold 2: 2.4579719016246236e-07
[GPBoost] [Warning] The 'objective' (='binary') for boosting and the 'likelihood' (='gaussian') for the GPModel do not match. It is assumed tha

Androids Analysis

In [11]:
# Paths
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import wilcoxon

dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/androids_model_dataset_basic.csv")
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/GradBoost/Androids")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Load
df = pd.read_csv(dataset)

meta_cols = [
    "file_path", "file", "file_stem", "bdi_score",
    "depressed", "fold", "speech_type", "subgroup_from_path"
]

feature_cols = [c for c in df.columns if c not in meta_cols]

df = df.dropna(subset=feature_cols + ["depressed", "file_stem"]).copy()

X = df[feature_cols].values
y = df["depressed"].astype(int).values
groups = df["file_stem"].values

# Train/test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_full, X_test = X[train_idx], X[test_idx]
y_train_full, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full)
X_test = scaler.transform(X_test)

# CV on training set
gkf = GroupKFold(n_splits=5)
cv_results = []

for fold, (cv_train_idx, cv_val_idx) in enumerate(
    gkf.split(X_train_full, y_train_full, groups=groups_train), start=1
):
    X_train, X_val = X_train_full[cv_train_idx], X_train_full[cv_val_idx]
    y_train, y_val = y_train_full[cv_train_idx], y_train_full[cv_val_idx]

    model = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    proba = model.predict_proba(X_val)[:, 1]

    dummy = DummyClassifier(strategy="stratified", random_state=42)
    dummy.fit(X_train, y_train)
    p_dum = dummy.predict_proba(X_val)[:, 1]
    e1 = np.abs(y_val - proba)
    e2 = np.abs(y_val - p_dum)
    if len(e1) >= 2 and not np.allclose(e1, e2):
        w_p = float(wilcoxon(e1, e2, zero_method="wilcox", mode="auto").pvalue)
    else:
        w_p = float("nan")
    print(f"  Wilcoxon p (|y - p_GBC| vs |y - p_dummy|) fold {fold}:", w_p)

    cv_results.append({
        "fold": fold,
        "accuracy": accuracy_score(y_val, preds),
        "f1": f1_score(y_val, preds),
        "roc_auc": roc_auc_score(y_val, proba),
        "wilcoxon_p_vs_stratified_dummy": w_p
    })

cv_results_df = pd.DataFrame(cv_results)
print("CV fold results:")
print(cv_results_df)

cv_summary_df = pd.DataFrame([{
    "subset": "cv_train",
    "n_rows": len(train_idx),
    "n_depressed": int(y_train_full.sum()),
    "n_control": int((y_train_full == 0).sum()),
    "accuracy_mean": cv_results_df["accuracy"].mean(),
    "accuracy_std": cv_results_df["accuracy"].std(),
    "f1_mean": cv_results_df["f1"].mean(),
    "f1_std": cv_results_df["f1"].std(),
    "roc_auc_mean": cv_results_df["roc_auc"].mean(),
    "roc_auc_std": cv_results_df["roc_auc"].std(),
}])

print("\nCV summary:")
print(cv_summary_df)

# Final model on full training set
final_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

final_model.fit(X_train_full, y_train_full)
test_preds = final_model.predict(X_test)
test_proba = final_model.predict_proba(X_test)[:, 1]

test_summary_df = pd.DataFrame([{
    "subset": "held_out_test",
    "n_rows": len(test_idx),
    "n_depressed": int(y_test.sum()),
    "n_control": int((y_test == 0).sum()),
    "accuracy": accuracy_score(y_test, test_preds),
    "f1": f1_score(y_test, test_preds),
    "roc_auc": roc_auc_score(y_test, test_proba)
}])

print("\nHeld-out test results:")
print(test_summary_df)

dummy_t = DummyClassifier(strategy="stratified", random_state=42)
dummy_t.fit(X_train_full, y_train_full)
p_dum_t = dummy_t.predict_proba(X_test)[:, 1]
e_gb = np.abs(y_test - test_proba)
e_du = np.abs(y_test - p_dum_t)
if len(e_gb) >= 2 and not np.allclose(e_gb, e_du):
    w_held = float(wilcoxon(e_gb, e_du, zero_method="wilcox", mode="auto").pvalue)
else:
    w_held = float("nan")
print("\nWilcoxon p (held-out, |y - p_GBC| vs |y - p_dummy|):", w_held)

pd.DataFrame([{"wilcoxon_p_vs_stratified_dummy_heldout": w_held}]).to_csv(
    RESULTS_PATH / "androids_gbc_wilcoxon_heldout.csv", index=False
)

cv_results_df.to_csv(RESULTS_PATH / "androids_gbc_cv_folds.csv", index=False)
cv_summary_df.to_csv(RESULTS_PATH / "androids_gbc_cv_summary.csv", index=False)
test_summary_df.to_csv(RESULTS_PATH / "androids_gbc_test_summary.csv", index=False)

  Wilcoxon p (|y - p_GBC| vs |y - p_dummy|) fold 1: 0.13144588227418502
  Wilcoxon p (|y - p_GBC| vs |y - p_dummy|) fold 2: 0.7773074483961245
  Wilcoxon p (|y - p_GBC| vs |y - p_dummy|) fold 3: 0.15275062637845202
  Wilcoxon p (|y - p_GBC| vs |y - p_dummy|) fold 4: 0.6944577103650468
  Wilcoxon p (|y - p_GBC| vs |y - p_dummy|) fold 5: 0.8058925247934928
CV fold results:
   fold  accuracy        f1   roc_auc  wilcoxon_p_vs_stratified_dummy
0     1  0.444444  0.500000  0.467532                        0.131446
1     2  0.666667  0.727273  0.725000                        0.777307
2     3  0.722222  0.705882  0.793750                        0.152751
3     4  0.638889  0.763636  0.492308                        0.694458
4     5  0.628571  0.682927  0.640523                        0.805893

CV summary:
     subset  n_rows  n_depressed  n_control  accuracy_mean  accuracy_std  \
0  cv_train     179          101         78       0.620159      0.104734   

    f1_mean    f1_std  roc_auc_mean  roc

In [12]:
# Paths
from pathlib import Path
import numpy as np
import pandas as pd
import gpboost as gpb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import wilcoxon

dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/androids_model_dataset_basic.csv")
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/GradBoost/Androids")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Load
df = pd.read_csv(dataset)

meta_cols = [
    "file_path", "file", "file_stem", "bdi_score",
    "depressed", "fold", "speech_type", "subgroup_from_path"
]

feature_cols = [c for c in df.columns if c not in meta_cols]

df = df.dropna(subset=feature_cols + ["depressed", "file_stem"]).copy()
df["group_code"] = pd.factorize(df["file_stem"])[0].astype(np.int32)

X = df[feature_cols].values
y = df["depressed"].astype(int).values
groups = df["group_code"].values

# Train/test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_full, X_test = X[train_idx], X[test_idx]
y_train_full, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full)
X_test = scaler.transform(X_test)

# CV on training set
gkf = GroupKFold(n_splits=5)
fold_results = []

for fold, (cv_train_idx, cv_val_idx) in enumerate(
    gkf.split(X_train_full, y_train_full, groups=groups_train), start=1
):
    X_train, X_val = X_train_full[cv_train_idx], X_train_full[cv_val_idx]
    y_train, y_val = y_train_full[cv_train_idx], y_train_full[cv_val_idx]
    group_train = groups_train[cv_train_idx].astype(np.int32)
    group_val = groups_train[cv_val_idx].astype(np.int32)

    gp_model = gpb.GPModel(group_data=group_train)
    train_data = gpb.Dataset(X_train, label=y_train)

    params = {
        "objective": "binary",
        "learning_rate": 0.05,
        "max_depth": 4,
        "verbose": 0
    }

    gpbst = gpb.train(
        params=params,
        train_set=train_data,
        gp_model=gp_model,
        num_boost_round=200
    )

    pred = gpbst.predict(
        data=X_val,
        group_data_pred=group_val,
        predict_var=False
    )

    p_gp = pred["response_mean"]
    preds = (p_gp >= 0.5).astype(int)

    gbc = GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
    )
    gbc.fit(X_train, y_train)
    p_gbc = gbc.predict_proba(X_val)[:, 1]
    e_gbc = np.abs(y_val - p_gbc)
    e_gp = np.abs(y_val - p_gp)
    if len(e_gbc) >= 2 and not np.allclose(e_gbc, e_gp):
        w_pair = float(wilcoxon(e_gbc, e_gp, zero_method="wilcox", mode="auto").pvalue)
    else:
        w_pair = float("nan")
    print(f"  Wilcoxon p (|y - p_GBC| vs |y - p_GP|) fold {fold}:", w_pair)

    fold_results.append({
        "fold": fold,
        "accuracy": accuracy_score(y_val, preds),
        "f1": f1_score(y_val, preds),
        "roc_auc": roc_auc_score(y_val, p_gp),
        "wilcoxon_p_gbc_vs_gpboost": w_pair
    })

fold_results_df = pd.DataFrame(fold_results)
print("CV fold results:")
print(fold_results_df)

cv_summary_df = pd.DataFrame([{
    "subset": "cv_train",
    "n_rows": len(train_idx),
    "n_depressed": int(y_train_full.sum()),
    "n_control": int((y_train_full == 0).sum()),
    "accuracy_mean": fold_results_df["accuracy"].mean(),
    "accuracy_std": fold_results_df["accuracy"].std(),
    "f1_mean": fold_results_df["f1"].mean(),
    "f1_std": fold_results_df["f1"].std(),
    "roc_auc_mean": fold_results_df["roc_auc"].mean(),
    "roc_auc_std": fold_results_df["roc_auc"].std()
}])

print("\nCV summary:")
print(cv_summary_df)

# Final model on full training set
gp_model_final = gpb.GPModel(group_data=groups_train.astype(np.int32))
train_data_final = gpb.Dataset(X_train_full, label=y_train_full)

params = {
    "objective": "binary",
    "learning_rate": 0.05,
    "max_depth": 4,
    "verbose": 0
}

gpbst_final = gpb.train(
    params=params,
    train_set=train_data_final,
    gp_model=gp_model_final,
    num_boost_round=200
)

test_pred = gpbst_final.predict(
    data=X_test,
    group_data_pred=groups_test.astype(np.int32),
    predict_var=False
)

test_proba = test_pred["response_mean"]
test_preds = (test_proba >= 0.5).astype(int)

test_summary_df = pd.DataFrame([{
    "subset": "held_out_test",
    "n_rows": len(test_idx),
    "n_depressed": int(y_test.sum()),
    "n_control": int((y_test == 0).sum()),
    "accuracy": accuracy_score(y_test, test_preds),
    "f1": f1_score(y_test, test_preds),
    "roc_auc": roc_auc_score(y_test, test_proba)
}])

print("\nHeld-out test results:")
print(test_summary_df)

gbc_test = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
)
gbc_test.fit(X_train_full, y_train_full)
p_gbc_test = gbc_test.predict_proba(X_test)[:, 1]
e1h = np.abs(y_test - p_gbc_test)
e2h = np.abs(y_test - test_proba)
if len(e1h) >= 2 and not np.allclose(e1h, e2h):
    w_held = float(wilcoxon(e1h, e2h, zero_method="wilcox", mode="auto").pvalue)
else:
    w_held = float("nan")
print("\nWilcoxon p (held-out, |y - p_GBC| vs |y - p_GP|):", w_held)
if w_held < 0.05:
    print("  GBC and GPBoost calibration errors differ significantly (alpha=0.05) on this test set.")
else:
    print("  No significant difference between GBC and GPBoost at alpha=0.05 (difference may be chance / test-set specific).")

pd.DataFrame([{"wilcoxon_p_gbc_vs_gpboost_heldout": w_held}]).to_csv(
    RESULTS_PATH / "androids_gpboost_wilcoxon_heldout.csv", index=False
)

fold_results_df.to_csv(RESULTS_PATH / "androids_gpboost_cv_folds.csv", index=False)
cv_summary_df.to_csv(RESULTS_PATH / "androids_gpboost_cv_summary.csv", index=False)
test_summary_df.to_csv(RESULTS_PATH / "androids_gpboost_test_summary.csv", index=False)

[GPBoost] [Warning] The 'objective' (='binary') for boosting and the 'likelihood' (='gaussian') for the GPModel do not match. It is assumed that the 'objective' for boosting is correctly specified, and the likelihood of the GPModel is changed accordingly. This can be problematic if the GPModel has been pre-trained 
  Wilcoxon p (|y - p_GBC| vs |y - p_GP|) fold 1: 0.0005046432957481209
[GPBoost] [Warning] The 'objective' (='binary') for boosting and the 'likelihood' (='gaussian') for the GPModel do not match. It is assumed that the 'objective' for boosting is correctly specified, and the likelihood of the GPModel is changed accordingly. This can be problematic if the GPModel has been pre-trained 
  Wilcoxon p (|y - p_GBC| vs |y - p_GP|) fold 2: 0.030108384222708284
[GPBoost] [Warning] The 'objective' (='binary') for boosting and the 'likelihood' (='gaussian') for the GPModel do not match. It is assumed that the 'objective' for boosting is correctly specified, and the likelihood of the G